In [27]:
from dotenv import load_dotenv
import os

load_dotenv()

REMOTE_MCP_URL = os.getenv("REMOTE_MCP_URL")
TOOLSETS = "repos,issues,pull_requests,context,users,git"
SERVER_NAME = os.getenv("SERVER_NAME")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

In [28]:
def build_github_connection() -> dict:
    headers = {
        "X-MCP-Toolsets": TOOLSETS,
        "X-MCP-Readonly": "true",
        "Authorization": f"Bearer {GITHUB_TOKEN}"
    }

    return {
        SERVER_NAME:{
            "url": REMOTE_MCP_URL,
            "headers": headers,
            "transport": "http",
            "timeout": 60,
            "sse_read_timeout": 300
        }
    }

In [29]:
CONNECTIONS = build_github_connection()

In [31]:
from langchain_mcp_adapters.client import MultiServerMCPClient

In [32]:
mcp_client = MultiServerMCPClient(CONNECTIONS)

In [33]:
async def fetch_catalog(client, server_name: str | None):
    async with client.session(server_name) as session:
        tools = await session.list_tools()
        print(tools)

    return None

In [34]:
await fetch_catalog(mcp_client, SERVER_NAME)

meta=None nextCursor=None tools=[Tool(name='get_commit', title=None, description='Get details for a commit from a GitHub repository', inputSchema={'type': 'object', 'properties': {'detail': {'type': 'string', 'description': 'Level of detail to include for changed files. "none" omits stats and files entirely. "stats" (default) includes per-file metadata: filename, status, and lines-of-code counts (additions, deletions, changes), with no patch content. "full_patch" additionally includes the unified diff content for each file and can be very large.', 'default': 'stats', 'enum': ['none', 'stats', 'full_patch']}, 'owner': {'type': 'string', 'description': 'Repository owner', 'x-mcp-header': 'owner'}, 'page': {'type': 'number', 'description': 'Page number for pagination (min 1)', 'minimum': 1}, 'perPage': {'type': 'number', 'description': 'Results per page for pagination (min 1, max 100)', 'minimum': 1, 'maximum': 100}, 'repo': {'type': 'string', 'description': 'Repository name', 'x-mcp-head

In [35]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

In [36]:
model = ChatOpenAI(
    model= "openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [37]:
tools = await mcp_client.get_tools(server_name=SERVER_NAME)
tools

[StructuredTool(name='get_commit', description='Get details for a commit from a GitHub repository', args_schema={'type': 'object', 'properties': {'detail': {'type': 'string', 'description': 'Level of detail to include for changed files. "none" omits stats and files entirely. "stats" (default) includes per-file metadata: filename, status, and lines-of-code counts (additions, deletions, changes), with no patch content. "full_patch" additionally includes the unified diff content for each file and can be very large.', 'default': 'stats', 'enum': ['none', 'stats', 'full_patch']}, 'owner': {'type': 'string', 'description': 'Repository owner', 'x-mcp-header': 'owner'}, 'page': {'type': 'number', 'description': 'Page number for pagination (min 1)', 'minimum': 1}, 'perPage': {'type': 'number', 'description': 'Results per page for pagination (min 1, max 100)', 'minimum': 1, 'maximum': 100}, 'repo': {'type': 'string', 'description': 'Repository name', 'x-mcp-header': 'repo'}, 'sha': {'type': 'str

In [38]:
system_prompt = """
You are a github assistant. Use mcp tools for live data. Do not create any PRs or repos. 
"""

agent = create_agent(
    model,
    tools=tools,
    system_prompt=system_prompt
)

In [39]:
question="Who am I on github? Name a few repositories if available"

output = await agent.ainvoke({
    "messages": [
        {"role": "user", "content": question},
    ],
})

In [45]:
print(output["messages"][-1].content)

**GitHub profile**

| Field | Value |
|-------|-------|
| **Login** | `hp1430` |
| **Name** | Himanshu Parashar |
| **Company** | Accenture |
| **Email** | parasharhimanshu41@gmail.com |
| **Location** | Gurugram |
| **Bio** | 👋 Hi, I'm Himanshu! <br>💻 MERN developer with expertise in Node.js and React.js <br>Currently working as a Packaged Application Development Associate at Accenture. |
| **Public repos** | 42 |
| **Followers** | 2 |
| **Following** | 1 |

**Selected public repositories**

| Repo | Description | Language |
|------|-------------|----------|
| [`My-DSA-codes-in-C-`](https://github.com/hp1430/My-DSA-codes-in-C-) | A collection of DSA (Data‑Structure & Algorithm) solutions written in C++. | C++ |
| [`Mail-Scheduler`](https://github.com/hp1430/Mail-Scheduler) | A simple mail‑scheduler utility, implemented in JavaScript. | JavaScript |
| [`nginx-config`](https://github.com/hp1430/nginx-config) | Sample NGINX configuration snippets. | N/A (plain text) |
| [`Instagram-Reels

In [47]:
output = await agent.ainvoke({
    "messages": [
        {"role": "user", "content": "Can you tell my most recent repository"}
    ],
})

In [48]:
print(output["messages"][-1].content)

Here is the most recently created or updated repository in your GitHub account:

| Repository | Description | Language | Updated | Created |
|------------|-------------|----------|---------|---------|
| **Ticketmaster_agent** | – | Python | 2026‑09‑23 T11:27:16Z | 2026‑09‑19 T11:16:05Z |

**Link:** <https://github.com/hp1430/Ticketmaster_agent>  

If you’d like more details (e.g., README contents, list of files, recent commits), just let me know and I can fetch those for you.
